<a href="https://colab.research.google.com/github/devdyuthi/llm-learning-projs/blob/main/week2%26week3clipper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate
!pip install -q transformers==4.44.2

In [ ]:
import os
import torch
from IPython.display import Markdown, display
from google.colab import drive, userdata
from huggingface_hub import login
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig

# 1. Connect Google Drive
drive.mount("/content/drive")
audio_filename = "/content/drive/MyDrive/llms/denver_extract.mp3"

# 2. Define Model
LLAMA = "meta-llama/Llama-3.2-3B-Instruct"

# 3. Authenticate with Hugging Face
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# =====================================================================
# STEP 1: Transcribe Audio with Word-Level Timestamps
# =====================================================================
print("⏳ Loading Whisper Transcription Model...")
pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium.en",
    torch_dtype=torch.float16,
    device='cuda:0',
    return_timestamps=True
)

print("🎙️ Transcribing audio (this may take a moment depending on file length)...")
result = pipe(audio_filename)

formatted_chunks = []
for chunk in result["chunks"]:
    start, end = chunk["timestamp"]


    start_str = f"{start:.1f}s" if start is not None else "0.0s"
    end_str = f"{end:.1f}s" if end is not None else "end"


    formatted_chunks.append(f"[{start_str} - {end_str}]: {chunk['text'].strip()}")

transcription_with_timestamps = "\n".join(formatted_chunks)

print("\n✅ Transcription Complete! Timeline Preview:")
print("\n".join(formatted_chunks[:5]))
print("...")

⏳ Loading Whisper Transcription Model...
🎙️ Transcribing audio (this may take a moment depending on file length)...


/usr/local/lib/python3.12/dist-packages/transformers/models/whisper/generation_whisper.py:496: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



✅ Transcription Complete! Timeline Preview:
[0.0s - 2.6s]: and kind of the confluence of this whole idea
[2.6s - 5.1s]: of the Confluence Week, the merging of two rivers.
[5.1s - 9.6s]: And as we've kind of seen recently in politics
[9.6s - 13.8s]: and in the world, there's a lot of situations
[13.8s - 15.7s]: where water is very important right now.
...


In [ ]:
# =====================================================================
# STEP 2: Analyze & Clip Recommendation Engine (FINAL WORKING VERSION)
# =====================================================================

import gc
import torch
from transformers import PreTrainedModel, BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM, TextStreamer
from IPython.display import Markdown, display


PreTrainedModel.to = torch.nn.Module.to
gc.collect()
torch.cuda.empty_cache()

system_message = """
You are an expert short-form video editor and social media growth strategist.
Your job is to analyze video transcripts with timestamps and find high-engagement segments
that will perform incredibly well as standalone YouTube Shorts or Instagram Reels.
"""

user_prompt = f"""
Below is the timestamped transcript of our uploaded video.
Please look for the absolute best 30 to 60-second segment that has a strong hook in the first 3 seconds and delivers a complete, satisfying thought.

Provide your output in clean Markdown format with the following sections:
- **Recommended Clip Timestamp**: (State the exact Start Time and End Time)
- **The Hook**: (Quote the opening line of the clip)
- **Viral Reasoning**: (Explain why this specific part will hold attention on Instagram/YouTube)
- **Platform Copy**:
  - A search-optimized title for YouTube Shorts.
  - A high-engagement, hashtag-rich caption for Instagram Reels.

Transcription:
{transcription_with_timestamps}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
]

print(" Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

print("Loading Llama-3.2 (4-bit Quantized)...")
model = AutoModelForCausalLM.from_pretrained(
    LLAMA,
    device_map="auto",
    quantization_config=quant_config,
    torch_dtype=torch.bfloat16
)

# Render message mapping schema into raw evaluation token ids
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt")


gpu_inputs = inputs.cuda()

# Initialize text streamer
streamer = TextStreamer(tokenizer, skip_prompt=True)

print("\nAnalyzing transcript for the best viral clip...\n")
print("-" * 50)

# Execute inference on the GPU-allocated text tokens
outputs = model.generate(
    input_ids=gpu_inputs,
    max_new_tokens=1500,
    streamer=streamer,
    pad_token_id=tokenizer.eos_token_id
)

print("-" * 50)

input_length = inputs.shape[1]
clean_response_tokens = outputs[0][input_length:]
final_markdown_text = tokenizer.decode(clean_response_tokens, skip_special_tokens=True)

print("\nDashboard \n")
display(Markdown(final_markdown_text))

🧠 Loading Tokenizer...
🚀 Loading Llama-3.2 (4-bit Quantized)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


🎬 Analyzing transcript for the best viral clip...

--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


<|start_header_id|>assistant<|end_header_id|>

**Recommended Clip Timestamp**
[14.9s - 21.0s]: October 9th. Please rise with the Pledge of Allegiance by Councilman Lopez.

**The Hook**
"Please rise with the Pledge of Allegiance by Councilman Lopez."

**Viral Reasoning**
This segment has a strong hook in the first three seconds, grabbing the viewer's attention with the Pledge of Allegiance. The clip then transitions into a meaningful discussion about Indigenous Peoples Day, highlighting the importance of inclusivity and respect for marginalized communities. The segment's themes of unity and social responsibility are likely to resonate with viewers, making it a compelling and shareable clip.

**Platform Copy**

**YouTube Shorts Title:** "Indigenous Peoples Day: A Celebration of Inclusivity and Respect"

**YouTube Shorts Description:** "Join us as we celebrate Indigenous Peoples Day and honor the contributions of Native American communities to our city and country. Hear from Councilman Lo

assistant

**Recommended Clip Timestamp**
[14.9s - 21.0s]: October 9th. Please rise with the Pledge of Allegiance by Councilman Lopez.

**The Hook**
"Please rise with the Pledge of Allegiance by Councilman Lopez."

**Viral Reasoning**
This segment has a strong hook in the first three seconds, grabbing the viewer's attention with the Pledge of Allegiance. The clip then transitions into a meaningful discussion about Indigenous Peoples Day, highlighting the importance of inclusivity and respect for marginalized communities. The segment's themes of unity and social responsibility are likely to resonate with viewers, making it a compelling and shareable clip.

**Platform Copy**

**YouTube Shorts Title:** "Indigenous Peoples Day: A Celebration of Inclusivity and Respect"

**YouTube Shorts Description:** "Join us as we celebrate Indigenous Peoples Day and honor the contributions of Native American communities to our city and country. Hear from Councilman Lopez and others as we discuss the importance of inclusivity and respect for marginalized communities. #IndigenousPeoplesDay #Inclusivity #Respect"

**Instagram Reels Caption:** "Celebrating Indigenous Peoples Day with love, respect, and inclusivity. Join us as we honor the contributions of Native American communities to our city and country. #IndigenousPeoplesDay #Inclusivity #Respect"